<a href="https://colab.research.google.com/github/yagocampello-ipnet/agrovale-ingestao/blob/main/scripts/tss/notebook_tss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS E AUTENTICAÇÃO
# ==============================================================================
!pip install -q gspread pandas google-cloud-bigquery db-dtypes google-genai

import time
import google.auth
from google.colab import auth
from google import genai
import gspread
from google.cloud import bigquery
import pandas as pd

auth.authenticate_user()
credentials, project_id = google.auth.default(
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
        "https://www.googleapis.com/auth/bigquery",
        "https://www.googleapis.com/auth/cloud-platform",
    ]
)

gc = gspread.authorize(credentials)
bq_client = bigquery.Client(credentials=credentials, project="agrovale-gcp")
client_gemini = genai.Client(
    api_key="Sua_Chave_API_Aqui",
)

print("1/4 - Autenticação concluída (BigQuery, Sheets e Gemini prontos)!")

# ==============================================================================
# 2. CONFIGURAÇÕES DA PLANILHA E TABELA GOLD
# ==============================================================================
URL_PLANILHA_TICKETS = "https://docs.google.com/spreadsheets/d/1BGZ20xf1oXhZbY4PrMyON9yPrpOB1MVXQ8Wklbb7bFM/edit?resourcekey=&gid=1250496843#gid=1250496843"  # <-- Mantenha a sua URL original aqui!
ABA_ORIGEM = "Form Responses 1"
ABA_DESTINO_SUGESTOES = "Sugestoes_IA"
TABELA_GOLD = "agrovale-gcp.gold.fato_quinzenal"

# ==============================================================================
# 3. CONSULTA AOS DADOS DE SATÉLITE NO BIGQUERY (GOLD)
# ==============================================================================
query_gold = f"SELECT * FROM `{TABELA_GOLD}`"
df_gold = bq_client.query(query_gold).to_dataframe(
    create_bqstorage_client=False
)
print(f"2/4 - Tabela Gold carregada: {len(df_gold)} registros encontrados.")

# ==============================================================================
# 4. LEITURA DOS CHAMADOS DA PLANILHA DE TICKETS
# ==============================================================================
planilha_tickets = gc.open_by_url(URL_PLANILHA_TICKETS)
aba_tickets = planilha_tickets.worksheet(ABA_ORIGEM)

dados_brutos = aba_tickets.get_all_values()
cabecalhos = dados_brutos[0]
df_tickets = pd.DataFrame(dados_brutos[1:], columns=cabecalhos)

# Remove colunas duplicadas mantendo a primeira
df_tickets = df_tickets.loc[:, ~df_tickets.columns.duplicated()]
print(f"3/4 - Tickets carregados: {len(df_tickets)} chamados encontrados.")

# ==============================================================================
# 5. CRUZAMENTO (LEFT JOIN) ENTRE TICKETS E DADOS DE SATÉLITE (ÚLTIMA QUINZENA)
# ==============================================================================
col_talhao_ticket = [
    c
    for c in df_tickets.columns
    if "talhão" in c.lower() or "talhao" in c.lower()
][0]
col_talhao_gold = "talhao_id"

df_tickets["chave_talhao"] = (
    df_tickets[col_talhao_ticket].astype(str).str.strip()
)
df_gold["chave_talhao"] = df_gold[col_talhao_gold].astype(str).str.strip()

# Ordena pela data mais recente e mantém apenas a última quinzena de cada talhão
df_gold_recente = df_gold.sort_values(
    by="data_inicio", ascending=False
).drop_duplicates(subset=["chave_talhao"])

tabela_final = pd.merge(
    df_tickets,
    df_gold_recente.drop(columns=[col_talhao_gold], errors="ignore"),
    on="chave_talhao",
    how="left",
)
tabela_final = tabela_final.drop(columns=["chave_talhao"], errors="ignore")

colunas_satelite = [
    c
    for c in df_gold_recente.columns
    if c not in [col_talhao_gold, "chave_talhao"]
]
for col in colunas_satelite:
  if col in tabela_final.columns:
    tabela_final[col] = (
        tabela_final[col]
        .astype("object")
        .fillna("Aguardando processamento de satélite")
    )

tabela_final = tabela_final.astype("object").fillna("")
# ==============================================================================
# 6. GERAR RESPOSTAS VIA GEMINI USANDO O THREAD ID INTERNO (COLUNA N)
# ==============================================================================
# Captura exatamente a Coluna N (índice 13) de cada linha da 'Form Responses 1'
threads_internos = [
    linha[13].strip() if len(linha) > 13 else "" for linha in dados_brutos[1:]
]
tabela_final["Thread_ID_Interno_Ref"] = threads_internos

try:
  aba_sugestoes = planilha_tickets.worksheet(ABA_DESTINO_SUGESTOES)
except gspread.exceptions.WorksheetNotFound:
  aba_sugestoes = planilha_tickets.add_worksheet(
      title=ABA_DESTINO_SUGESTOES, rows="100", cols="50"
  )

# Lê o cache das respostas já prontas cruzando pelo Thread ID Interno (ou Timestamp)
dados_existentes = aba_sugestoes.get_all_values()
cache_respostas = {}
if len(dados_existentes) > 1:
  cabecalho_existente = dados_existentes[0]
  idx_sugestao = (
      cabecalho_existente.index("Sugestao_Resposta_Gemini")
      if "Sugestao_Resposta_Gemini" in cabecalho_existente
      else -1
  )
  idx_thread = (
      cabecalho_existente.index("Thread_ID_Interno_Ref")
      if "Thread_ID_Interno_Ref" in cabecalho_existente
      else 0
  )

  if idx_sugestao != -1:
    for row in dados_existentes[1:]:
      chave = row[idx_thread].strip() if len(row) > idx_thread else ""
      texto_salvo = row[idx_sugestao].strip() if len(row) > idx_sugestao else ""
      if (
          chave
          and texto_salvo
          and not texto_salvo.startswith("=")
          and not texto_salvo.startswith("Erro")
      ):
        cache_respostas[chave] = texto_salvo

# Atualiza a aba Sugestoes_IA
aba_sugestoes.clear()
aba_sugestoes.update(
    range_name="A1",
    values=[tabela_final.columns.values.tolist()]
    + tabela_final.astype(str).values.tolist(),
)

respostas_geradas = []
modelos_tentativa = [
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash-lite",
]
colunas_ignorar = ["Thread_ID_Interno_Ref", "Sugestao_Resposta_Gemini"]

for idx, linha in tabela_final.iterrows():
  thread_id = str(linha.get("Thread_ID_Interno_Ref", "")).strip()
  chave_busca = thread_id if thread_id else str(linha.iloc[0]).strip()

  if chave_busca in cache_respostas:
    respostas_geradas.append([cache_respostas[chave_busca]])
    continue

  contexto_chamado = " | ".join([
      f"{col}: {val}"
      for col, val in linha.items()
      if str(val).strip() != ""
      and col not in colunas_ignorar
      and not str(col).startswith("Unnamed")
  ])[:3000]

  prompt = f"""
    Gere uma sugestão de resposta ao produtor rural em linguagem simples, direta e acolhedora, sem jargões técnicos.
    Cruze a dúvida do chamado com os dados de satélite abaixo.
    Se constar 'Aguardando processamento de satélite', informe educadamente que os dados daquele talhão ainda estão sendo processados.

    Dados do chamado e satélite:
    {contexto_chamado}
    """

  texto_final = ""
  for modelo_atual in modelos_tentativa:
    print(f"Linha {idx + 1}: tentando {modelo_atual}...")
    try:
      response = client_gemini.models.generate_content(
          model=modelo_atual,
          contents=prompt,
      )
      texto_final = response.text.strip()
      print(f"Linha {idx + 1}: sucesso com {modelo_atual}!")
      break
    except Exception as e:
      erro_tipo = type(e).__name__
      print(f"Linha {idx + 1}: falhou ({modelo_atual}) -> {erro_tipo}")
      if any(
          cod in str(e) for cod in ["429", "503", "404"]
      ) or "Timeout" in erro_tipo:
        time.sleep(2)
        continue
      texto_final = f"Erro ao gerar sugestão ({erro_tipo}): {e}"
      break
  else:
    texto_final = "Erro temporário (Cota/503/404) em todos os modelos. Execute novamente."

  respostas_geradas.append([texto_final])
  time.sleep(1)

# ==============================================================================
# 7. GRAVAR NA 'Sugestoes_IA' E NA COLUNA P DA 'Form Responses 1'
# ==============================================================================
total_linhas = len(tabela_final) + 1
col_destino_sugestoes = len(tabela_final.columns) + 1

if total_linhas >= 2:
  if aba_sugestoes.col_count < col_destino_sugestoes:
    aba_sugestoes.resize(cols=col_destino_sugestoes + 3)

  aba_sugestoes.update_cell(1, col_destino_sugestoes, "Sugestao_Resposta_Gemini")
  aba_sugestoes.update(
      range_name=f"{gspread.utils.rowcol_to_a1(2, col_destino_sugestoes)}:{gspread.utils.rowcol_to_a1(total_linhas, col_destino_sugestoes)}",
      values=respostas_geradas,
  )

  # Grava direto na Form Responses 1 (Coluna P) alinhado linha a linha com a Coluna N
  aba_tickets.update_acell("P1", "Sugestao_Resposta_Gemini")
  aba_tickets.update(
      range_name=f"P2:P{total_linhas}",
      values=respostas_geradas,
  )

print(
    "4/4 - 100% Automático! Respostas geradas via Vertex AI, vinculadas ao Thread ID (Coluna N) e gravadas na Coluna P!"
)

1/4 - Autenticação concluída (BigQuery, Sheets e Gemini prontos)!
2/4 - Tabela Gold carregada: 128 registros encontrados.
3/4 - Tickets carregados: 5 chamados encontrados.
Linha 3: tentando gemini-3.5-flash-lite...
Linha 3: sucesso com gemini-3.5-flash-lite!
Linha 4: tentando gemini-3.5-flash-lite...
Linha 4: sucesso com gemini-3.5-flash-lite!
Linha 5: tentando gemini-3.5-flash-lite...
Linha 5: falhou (gemini-3.5-flash-lite) -> ServerError
Linha 5: tentando gemini-3.5-flash-lite...
Linha 5: falhou (gemini-3.5-flash-lite) -> ServerError
Linha 5: tentando gemini-3.5-flash-lite...
Linha 5: sucesso com gemini-3.5-flash-lite!
4/4 - 100% Automático! Respostas geradas via Vertex AI, vinculadas ao Thread ID (Coluna N) e gravadas na Coluna P!
